# Chapitre 10 - Exercices 10.6 à 10.10

Ce notebook traite les exercices sur les jointures, l'asymétrie, Pregel et les sorties
des jobs :

- **10.6** : une jointure tri-fusion qui donne des résultats faux : trouver le bogue
  (raisonnement et code) ;
- **10.7** : choisir l'algorithme de jointure selon la taille et le partitionnement
  des entrées (calcul) ;
- **10.8** : une célébrité dans la jointure : clé chaude et salage (calcul et
  simulation) ;
- **10.9** : plus courts chemins en Pregel (mini-lab) ;
- **10.10** : écrire dans la base de production ou construire et basculer ? (calcul et
  raisonnement).



In [1]:
import heapq
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import batch as B
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")

## Exercice 10.6 : une jointure tri-fusion qui donne des résultats faux

**Énoncé.** Un collègue écrit la jointure du cours 3 (événements enrichis de l'année de
naissance de l'utilisateur) avec un tri secondaire. Sa version tourne sans erreur...
mais ne produit aucun événement enrichi.

```python
def mappeur(rec):
    if "birth" in rec:
        yield (rec["user_id"], 0, 0), rec["birth"]
    else:
        yield (rec["user_id"], 1, rec["ts"]), rec["url"]

def reducteur(cle, valeurs):        # cle = (user_id, étiquette, date)
    naissance = next(valeurs)
    for url in valeurs:
        yield [url, naissance]

B.run_mapreduce(entrees, mappeur, reducteur, n_reducers=4)      # pas de group_by
```

1. Exécuter ce job sur un petit jeu de données et constater le problème.
2. Expliquer précisément ce qui se passe (partitionnement, regroupement).
3. Corriger. Que faire des événements dont l'utilisateur n'existe pas dans l'export
   (jointure interne ou externe) ?

**Rappels.** Jointure tri-fusion côté reduce, tri secondaire : on trie sur la clé
complète, mais on **partitionne et regroupe** sur l'identifiant (cours 3, section 4,
section *Reduce-Side Joins and Grouping*).

In [2]:
users = B.make_users(200, seed=1)
events = B.make_events(2000, 210, n_urls=20, seed=2)          # quelques événements d'utilisateurs absents (200 à 209)
entrees = [users[:100], users[100:], events[:1000], events[1000:]]


def mappeur(rec):
    if "birth" in rec:
        yield (rec["user_id"], 0, 0), rec["birth"]
    else:
        yield (rec["user_id"], 1, rec["ts"]), rec["url"]


def reducteur_bogue(cle, valeurs):
    naissance = next(valeurs)
    for url in valeurs:
        yield [url, naissance]


rows = []
for R in [1, 4]:
    out, st = B.run_mapreduce(entrees, mappeur, reducteur_bogue, n_reducers=R)
    sortie = [x for o in out for x in o]
    rows.append({"version": f"boguée, {R} réducteur(s)", "groupes vus par les réducteurs": st.reduce_groups,
                 "événements en sortie": len(sortie),
                 "« naissances » qui sont des URL": sum(isinstance(x[1], str) for x in sortie)})
pd.DataFrame(rows).set_index("version")

,groupes vus par les réducteurs,événements en sortie,« naissances » qui sont des URL
version,,,
"boguée, 1 réducteur(s)",2200,0,0
"boguée, 4 réducteur(s)",2200,0,0


**Solution.**

*2.* Sans `group_by`, la clé de **regroupement** est la clé complète (identifiant,
étiquette, date) : chaque événement forme son propre groupe, d'un seul élément. Le
réducteur appelle `next(valeurs)` qui consomme cet unique élément comme « naissance »,
puis la boucle ne trouve rien : tous les événements disparaissent (seuls d'éventuels
événements d'un même utilisateur au même horodatage formeraient un groupe de deux, et
leur « naissance » serait une URL). Dans `B.run_mapreduce`, le partitionnement suit la
clé de regroupement ; dans Hadoop, partitionneur et comparateur de regroupement se
configurent séparément, et oublier le partitionneur envoie l'utilisateur et ses
événements chez des réducteurs différents : le résultat dépend alors du nombre de
réducteurs, ce qui est encore plus difficile à diagnostiquer.

*3.* Il faut partitionner et regrouper sur `cle[0]` (paramètre `group_by`), et vérifier
que la première valeur est bien une naissance : si l'utilisateur est absent de l'export,
le premier élément est déjà un événement. Jointure **interne** : on ignore ces
événements ; jointure **externe gauche** : on les garde avec une naissance inconnue.
Le choix est métier, mais il doit être explicite (et compté).

In [3]:
def reducteur_externe(uid, valeurs):
    premier = next(valeurs)
    if isinstance(premier, int):           # c'est bien la naissance
        naissance = premier
    else:                                  # utilisateur absent de l'export : jointure externe gauche
        naissance = None
        yield [premier, None]
    for url in valeurs:
        yield [url, naissance]


out, st = B.run_mapreduce(entrees, mappeur, reducteur_externe, n_reducers=4, group_by=lambda k: k[0])
sortie = [x for o in out for x in o]
orphelins = sum(x[1] is None for x in sortie)
attendus = sum(e["user_id"] >= 200 for e in events)
ref = B.sort_merge_join(events, users, "user_id")
pd.DataFrame({"valeur": [len(sortie), len(events), orphelins, attendus, len(ref)]},
             index=["événements en sortie (corrigé)", "événements en entrée", "dont sans utilisateur (naissance inconnue)",
                    "événements d'utilisateurs absents (attendu)", "jointure interne de référence"])

,valeur
événements en sortie (corrigé),2000
événements en entrée,2000
dont sans utilisateur (naissance inconnue),103
événements d'utilisateurs absents (attendu),103
jointure interne de référence,1897


La version corrigée restitue les 2 000 événements, dont ceux des utilisateurs absents
marqués « inconnu » ; la jointure interne de référence en garde le complément exact.

> **À retenir.** Dans un tri secondaire, la clé de tri, la clé de partitionnement et la
> clé de regroupement sont **trois choses distinctes** : on trie sur la clé composite, on
> partitionne et on regroupe sur la clé de jointure. Et l'on décide explicitement du
> sort des enregistrements sans correspondance.